Try the method of warping the best matching template to match ZTF lightcurves.

Steps:
- Pick (random select) a BTS non-Ia SN and retrieve the S params (and color).
- Possibly min criteria on nbr bands and data quality (later flag?)
- Parse S parameters of templates and find the closes match (within the selected narrow/wide/all classes).
- Find the peak date and subtract. Probably have to use salt, since parsnip does not seem to return a distinct peak prediction. Well kind of, but unclear what it means vs different bands. At the other hand salt often fails. Ok - so have to determine a global peak lightcurve, since we will need to fix that for the model we wish to modify.
Or can we do it per eff wavelength? We can find the peak time here as well as for the model.
So we first find the peak per band, and set this to zero. We then look at this effective wavelength of the model, and find the time of peak here and shift to this. 

- For each band, determine the effective wavelength.  
- From template, extract prediction at effective wavelength and lightcure times of sn, taking peak time and redshift (both aspects) into account.. 
- Divide sn with template, get surface modifier. 
- Do the same for all filters.
- Construct 2d interpolate based on surface modifiers to which we add ones at the edges for regulation. 
- Interpolate correlation surface to the same dimensions as the original template.
- Possibly limit dimensions. 
- Divide template with correction, save.


For X closest templates, fit with the different sncosmo templates, then draw which one to scale from based on how different the chi2dof values are. So if there are good matches, these will be used but if only bad exists, use there. 

Also, will limit generated templates to prob 50 days post peak (and from -20)

Add some intrinsic dispersion? Fit it somehow from the data? For each class, get the best fit templates to the best observed sne and see what intrinsic dispersion is needed to reach chi2/dof in some reasonable parameter range. 

In [ ]:
import pymongo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sncosmo
import pickle
import json
import parsnip
import lcdata
import astropy
from ampel.ztf.util.ZTFIdMapper import ZTFIdMapper
from ampel.ztf.view.ZTFFPTabulator import ZTFFPTabulator 
from sncosmo.fitting import DataQualityError

In [ ]:
# Parameters
snix = 9
scatter_sigma = 0.   # How much should we shift positions in parsnip space? 
parsnip_disperson = 0.0  # Additional scatter for more variability - not sure needed
include_sigma = 5.  # Reject outlying datapoints

In [ ]:
# Semi permanent
# Input supernova data
df_sn = pd.read_csv('/Users/jnordin/tmp/bts_poormatch_goodlc.csv')
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')
# parsnip
ppath = '/Users/jnordin/data/models/parsnip/v3_Jul2025/noiztf_model_hybrid_snia_scale2_10_unbalanced_z0_sncut_subsamp0.9_nocadscale_seed0.pt'

### 1. Retrieve a SN and its class, inspect parsnip fit. 

In [ ]:
# Seems like we now subsample from a list of poor fit objects, extend later
sndata = dict(df_sn.iloc[snix])
z = float(list(df_bts[df_bts['ZTFID']==sndata['mod']]['redshift'])[0])
sndata

#### Get photometric (baseline corrected) from db

In [ ]:
tabulators = []
tabulators.append(
    ZTFFPTabulator(inclusion_sigma=include_sigma)    
)
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts
def get_db_table( name, database, tabulators=[] ):
    """
    For ZTF name, get photopoints and then tables. 
    """

    # Name
    if isinstance(name, int):    
        print('Assuming name given as stock')
        stock = int
    elif re.search('ZTF', name):
        stock = ZTFIdMapper.to_ampel_id(name)
    else:
        print('Cannot parse', name)
        return None 

    # Obtain photopoints
    dps = [dp for dp in database.t0.find({'stock':stock})]

    # Convert to table(s)
    ftables = []
    for tabulator in tabulators:
        ftables.append( tabulator.get_flux_table(dps) )
    if len(ftables)>1:
        print('Implement astropy table appending!')
    return ftables.pop(0)

In [ ]:
type(db)

In [ ]:
tab = get_db_table(sndata['mod'], database=db, tabulators=tabulators)
tab.sort('time')

In [ ]:
tab.meta = {
            'object_id':sndata['mod'],
            'type':sndata['class'],
            'redshift':z,
        }

In [ ]:
# Do sncosmo fit (to get peak)

In [ ]:
# create a model
model = sncosmo.Model(source='salt2')
#fitprop = ['t0', 'x0', 'x1','c']
fitprop = ['t0', 'x0', 'x1']
model.set(z=z)
model.set(c=0)

In [ ]:
result, fitted_model = sncosmo.fit_lc(
    btab, model,
    fitprop) 

In [ ]:
fig = sncosmo.plot_lc(btab, model=fitted_model, errors=result.errors)


#### Do parsnip fit 

In [ ]:
model = parsnip.load_model(ppath)

In [ ]:
# can add sampling to get finer time steps, if helps interpolation etc.
mfit = model.predict_light_curve(btab)

In [ ]:
parsnip.plot_light_curve(btab, model)

In [ ]:
plc = parsnip.preprocess_light_curve(tab,settings=model.settings)

In [ ]:
plc.meta

In [ ]:
# This is mentioned as how to convert to the time of the internal parsnip grid.
# internal time = (lctime-parsnip_reference_time)*SIDEREAL_TIME
# seems like a farily rough guess, basically peak light. 
SIDEREAL_SCALE = 86400. / 86164.0905
plc.meta['parsnip_reference_time']
# The actual fit also returns a "ref_times", which seems to be produced during the fit.
# Possibly this has to do with numerics in the fit. Still not clear whether to include or not
# 
# original time = grid_time / SIDEREAL_SCALE + reference_time 

Looking good? Inspect pulls? Anyway, if you feel model is good enough, continue. Will take the peak time and subtract from table.

In [ ]:
# In some cases this seems off (0)
mfit[2]['ref_times']

### Find three template models to compare with

In [ ]:
dft = pd.read_csv('/Users/jnordin/data/models/parsnip/v3_Jul2025/modelfits/sncosmotemplates_parsnip_ampcol.csv')

In [ ]:
# We do need some sncosmo template matching
sncosmo_modelmap = {
     'PopIII':'PopIII',
     'SN II':'SN II',
     'SN II-pec':'SN II',
     'SN IIL':'SN II',
     'SN IIL/P':'SN II',
     'SN IIP':'SN II',
     'SN IIb':'SN II',
     'SN IIn':'SN II',
     'SN Ia':'SN Ia',
     'SN Ib':'SN Ib/c',
     'SN Ib/c':'SN Ib/c',
     'SN Ic':'SN Ib/c',
     'SN Ic-BL': 'SN Ib/c',
    'SN Icn': 'SN Ib/c', 
    'SN Ia-CSM': 'SN Ia', 
    'SN Ic-pec': 'SN Ib/c', 
    'SN Ia-91bg': 'SN Ia', 
    'SN Ia-91T': 'SN Ia', 
    'SN Ca-rich-Ca': None, # No templates
    'SN Ibn': 'SN Ib/c', 
    'Ca-rich': None,
    'SN Ib/c': 'SN Ib/c', 
    'SN II-pec': 'SN II', 
    'ILRT': None,      # No templates 
    'SN Ia-pec': 'SN Ia', 
    'TDE': None,       # No templates 
    'SN Iax': 'SN Ia', 
    'SLSN-I': 'SN II',      # No templates available, using these for now
    'SN Ia-SC': 'SN Ia', 
    'SN IIn': 'SN II', 
    'SN IIn-pec': 'SN II', 
    'SN Ib-pec': 'SN Ib/c',
}

In [ ]:
dft['wideclass'] = dft['class'].map(sncosmo_modelmap)

In [ ]:
# Subselect to templates of the same 
if sum(dft['class']==sndata['class'])>0:
    print('Template exists for specific class - use this')
    dfclass = dft[dft['class']==sndata['class']]
else:
    print('No specific class, using wide class')
    dfclass = dft[dft['wideclass']==sndata['wideclass']]

In [ ]:
#dfclass = dft[dft['wideclass']==sndata['wideclass']]

In [ ]:
#dfclass = dft

In [ ]:
dfclass

In [ ]:
# Create taret hyperparameter position

In [ ]:
N1 = sndata['S1'] + sndata['dS1'] * np.random.normal(size=1, loc=0, scale=scatter_sigma) + np.random.normal(size=1, loc=0, scale=parsnip_disperson)
N2 = sndata['S2'] + sndata['dS2'] * np.random.normal(size=1, loc=0, scale=scatter_sigma) + np.random.normal(size=1, loc=0, scale=parsnip_disperson)
N3 = sndata['S3'] + sndata['dS3'] * np.random.normal(size=1, loc=0, scale=scatter_sigma) + np.random.normal(size=1, loc=0, scale=parsnip_disperson)

In [ ]:
print(sndata['S1'], sndata['S2'], sndata['S3'])

In [ ]:
print(N1,N2,N3)

In [ ]:
# Geometric difference
gdiff = (
            (N1-dfclass['S1'])**2 / (sndata['dS1']**2+dfclass['dS1']**2) + 
            (N2-dfclass['S2'])**2 / (sndata['dS2']**2+dfclass['dS2']**2) + 
            (N3-dfclass['S3'])**2 / (sndata['dS3']**2+dfclass['dS3']**2)  
)

In [ ]:
# Invert to relative properties
rprop = 1 / gdiff
rprop /= np.sum(rprop)

In [ ]:
# We now wish to select X (3?) objects from these, using these
# as some relative prop. What if we draw a uniform each and then multiply?
rprop

In [ ]:
rand = np.random.uniform(size=len(rprop))

In [ ]:
sel = rprop * rand

In [ ]:
sel

In [ ]:
sel.nlargest(n=3)

In [ ]:
im = list( sel.nlargest(n=5).index )

In [ ]:
im

In [ ]:
basemodels = dft.loc[im]

In [ ]:
#basemodels = dft

In [ ]:
basemodels

#### Fit the best model(s)

In [ ]:
models = {}
dust = sncosmo.CCM89Dust()
for _, row in basemodels.iterrows():
    if row['mod'] in ['salt22','salt2-2','salt3','salt2-4','salt2','salt2-1','salt20','salt21']:
        continue
    print('Adding', row['mod'])
    m = sncosmo.Model(source=row['mod'],
                      effects=[dust],
                      effect_names=['host'],
                      effect_frames=['rest'])
    m.set(hostr_v = 3.1)
    models[row['mod']] = {
        'model': m,
        'fitprop': ['t0', 'amplitude', 'hostebv'],
        'bounds': {
            'hostebv':[-3,3],
        },
        'class': row['class']
    }

In [ ]:
results = {modelname:[] for modelname in models.keys()}

In [ ]:
max_chidof = 50

In [ ]:
from scipy.interpolate import make_smoothing_spline
from scipy.interpolate import (
    InterpolatedUnivariateSpline as Spline1d,
    RectBivariateSpline as Spline2d
)
def get_template_correction( 
    datatable, templatename, z, 
    max_chidof=50, min_bands=2, min_point_band=5, pull_cut = 10,
    rv=3.1, max_phases = [-20,50], require_phasecoverage = True,
    spline_lam=0.1,
    plot_dir = None, plot_label='ZTFA',
):
    """
    Attempt to calculate correction coefficients for this template to the datatable.
    Note that we will create a template with the same mean color as the SN.

    - Fit the template to the data, iteratively rejecting flux and time outliers.
    - If fit not good enough (bands, datapoints, chi), end process.
    - Extract model prediction at the specific phases and wavelengths of each band. 
    - Divide this with the observe data to create a correction vector 
        V( restPhase, restWave )
        where restPhase are restframe phases of the observation in the template time
        and restWave the effective wavelength of the observed band shifted to z=0.
    - Make sure that these end at 1 at wavelength ends.
    - Add buffer vectors of 1 at template min and max phases. 
    - Create 2d python interpolation object based on the above.
    - Interpolate from object to the phases of the template.
    - Apply same reddening to template as was fitted.
    - Divide template with interpolation.

    if require_phasecoverage: 
    - Do not extrapolate phases beyond what the supernova template included. If false
    will use max_phases or template range (whichver is smaller). (Check not done per band).
    """

    # Fit template 
    print('... applying', templatename)
    if re.search( 'salt', templatename ):
        raise ValueError('SALT templates not incorporated')
    dust = sncosmo.CCM89Dust()
    m = sncosmo.Model(source=templatename,
                      effects=[dust],
                      effect_names=['host'],
                      effect_frames=['rest'])
    m.set(hostr_v = rv)
    m.set(z=z)
    fitprop = ['t0', 'amplitude', 'hostebv']
        
    # run the fit
    print('... data initially:',len(tab))
    mdict = {'model':templatename,'dps_init':len(tab)}
    try:
        result, fitted_model = sncosmo.fit_lc(
            tab, m,
            fitprop,  # parameters of model to vary
        )
        mdict.update( {
            result['param_names'][k]:result['parameters'][k] 
            for k in range(len(result['parameters'])) 
            }
        )

        # Look for outliers in time. 
        phases = (tab['time']-mdict['t0'])/(1+z)
        iGood = ( 
            ( max_phases[0] < phases ) & (phases < max_phases[1] ) & 
            ( phases > m.mintime() ) & 
            ( phases < m.maxtime() )
        )
        if sum(iGood)<len(phases):
            print('... outlying phases, remaning:', sum(iGood))

        # Look for mag outliers in flux
        result, fitted_model = sncosmo.fit_lc(
                tab[iGood], fitted_model,
                modelpar['fitprop']
        )
        iNorm = np.abs( 
            (tab['flux']-fitted_model.bandflux(tab['band'], tab['time'],zp=25,zpsys='ab')) / tab['fluxerr']
            ) < pull_cut 
        mdict['dps_fcut'] = sum(iNorm)
        if sum(iNorm)<len(phases):
            print('... outlying datapoints, remaning:', sum(iNorm))
        iTot = iGood & iNorm

        # Final fit
        result, fitted_model = sncosmo.fit_lc(
                tab[iTot], fitted_model,
                modelpar['fitprop']
        )
        mdict.update( {
            result['param_names'][k]:result['parameters'][k] 
            for k in range(len(result['parameters'])) 
            }
        )
        # Final outlier selection 
        phases = (tab['time']-mdict['t0'])/(1+z)
        iGood = ( 
            ( max_phases[0] < phases ) & (phases < max_phases[1] ) & 
            ( phases > m.mintime() ) & 
            ( phases < m.maxtime() )
        )
        mdict['dps_tcut'] = sum(iGood)
        iNorm = np.abs( 
            (tab['flux']-fitted_model.bandflux(tab['band'], tab['time'],zp=25,zpsys='ab')) / tab['fluxerr']
            ) < pull_cut 
        mdict['dps_fcut'] = sum(iNorm)
        iTot = iGood & iNorm
        mdict['dps_allcut'] = sum(iTot)
        print(' ... final dps', mdict['dps_allcut'] )
        
        

    except (RuntimeError, ValueError, KeyError, DataQualityError) as e:
        print('.. fit fail')
        mdict['success'] = False
        return mdict
    mdict['success'] = True 

    mdict.update( {k:result[k] for k in ['success', 'chisq', 'ndof', 'errors']} )
    mdict['absmag'] = fitted_model.source_peakabsmag(band='bessellb',magsys='ab')
    mdict['chidof'] = result.chisq / result.ndof 

    # Inspect whether fit is good enough for template construction
    mdict['lceval'] = {
        band: sum(tab['band'][iTot]==band)
        for band in set( tab['band'][iTot] )
    }
    if (len([count for count in mdict['lceval'].values()
            if count>min_point_band])) < min_bands:
        print('...not enough data')
        mdict['success'] = False
        return mdict         
    if mdict['chidof'] > max_chidof:
        print('...too poor fit')
        mdict['success'] = False
        return mdict

    # At this point we have a model which should be ok for template building. 

    # Create the frac
    startphase = max( m.mintime(), max_phases[0] )
    endphase = min( m.maxtime(), max_phases[1] )
    corr_frac = tab['flux'] / fitted_model.bandflux(tab['band'], tab['time'],zp=25,zpsys='ab')        
#    plt.figure()
    mdict['corrdata'] = {}
    for band in set( tab['band'][iTot] ):
        iBand = (tab['band'][iTot]==band)
        band_corr, band_err = list(corr_frac[iTot][iBand]), list(tab['fluxerr'][iTot][iBand])
        band_phase = list( (tab['time'][iTot][iBand]-mdict['t0'])/(1+z) )
        bandfunc = sncosmo.get_bandpass(band)
        rest_wave = bandfunc.wave_eff / (1+z)
        if require_phasecoverage:
            band_phase = band_phase
            band_corr = band_corr
            band_err = band_err
        else:
            band_phase = [startphase] + band_phase + [endphase]
            band_corr = [1] + band_corr + [1]
            band_err = [0.01] + band_err + [0.01]

        # Interpolate to model phases
        spl = make_smoothing_spline( band_phase, band_corr, 
                                    w=1.0 / np.array(band_err) ** 2, lam=spline_lam
                                    )
        dspl = make_smoothing_spline( band_phase, band_err, 
                                     w=1.0 / np.array(band_err) ** 2, lam=spline_lam
                                    )
        tphase = m.source._phase[
            (m.source._phase>=startphase) & (m.source._phase<=endphase)
        ]

        finterp = spl(tphase)

        mdict['corrdata'][band] = {
            'wave': rest_wave, 'phase': band_phase, 
            'frac': band_corr, 'err': band_err,
            'tphase':tphase, 'tcorr':finterp, 'terr':dspl(tphase)
        }
        if plot_dir:
            plt.plot(band_phase, band_corr, 'o', label="{} at {}".format(band, rest_wave))
            plt.plot(tphase, finterp, "-.", label="Interp: {}".format(band))
            plt.legend()
            plt.savefig(plot_dir + "/{}_{}_corr.png".format(plot_label,templatename))
    plt.show()

    # Create vectors for the interpolation
    # Wavelength limits 
    wave = [m.minwave()]
    phase, flux = None, None
    # Assuming ztf bands, doing this to ensure order
    for band in ['ztfg', 'ztfr', 'ztfi']:
        if not band in mdict['corrdata']:
            print('... no {} data for interpolation.'.format(band))
            continue
        wave.append( mdict['corrdata'][band]['wave'] )
        if phase is None:
            # Have to start initiating 
            phase = mdict['corrdata'][band]['tphase']
            flux = [np.ones( len(phase) )]
            flux.append( mdict['corrdata'][band]['tcorr'] )
        else:
            if sum( phase - mdict['corrdata'][band]['tphase'] )>0:
                raise ValueError('incompatible phase error')
            flux.append( mdict['corrdata'][band]['tcorr'] )
    wave.append( m.maxwave() )
    flux.append( np.ones( len(phase) ) )
    flux = np.array(flux).transpose()
    corrmodel = Spline2d(phase, wave, flux , kx=3, ky=3)

    # Apply reddening according to the fitted model to the flux
    flux = fitted_model.effects[0].propagate(np.array(wave), flux)
    #print('dust!')
    #foo = extinction.ccm89(np.array(wave), mdict['hostebv']*mdict['hostr_v'], mdict['hostr_v'] )
    #print(foo)
    #print('nover')
#    extinction.apply(extinction.ccm89(wave, ebv * r_v, r_v), flux)
#    dust.propagate(
    mdict['corrmodel'] = {'phase':phase,'wave':wave,'flux':flux}


    if plot_dir:
        # Plot the model
        plotwave = np.arange( 3500, 7000, 500 )
        print(plotwave)
        for plotphase in [-10,-5,0,5,10,15,20]:
            foo = corrmodel(plotphase, plotwave)
            print(foo[0])
            plt.plot(plotwave, corrmodel(plotphase, plotwave)[0], 
                     label='@ {}'.format(plotphase)
                    )
        plt.legend()
        plt.savefig(plot_dir + "/{}_interp.png".format(templatename)) 
        
    
    # Demo fig. Add panels to this
    if plot_dir:
        # Sncosmo figure
        fig = sncosmo.plot_lc(tab[iTot], model=fitted_model, errors=result.errors)
        fig.savefig(plot_dir + "/{}_{}_fit.png".format(plot_label,templatename)) 
        

    mdict['m'] = fitted_model
    mdict['om'] = m
    return mdict

In [ ]:
mdict = get_template_correction(tab, 'snana-2006jo',
                                       min_bands=2, min_point_band=5,
                                        require_phasecoverage=False,
                                       z=z, plot_dir='/Users/jnordin/tmp')

In [ ]:
mdict['corrmodel']

In [ ]:
from sncosmo import TimeSeriesSource
class WarpedTimeSeriesSource( TimeSeriesSource ):
    """
    Load existing TimeSeriesSource and warp according to input.
    """

    def __init__(self, phase, wave, flux, 
                 original_template_name, original_template_version=None,
                 time_spline_degree=3, name=None, version=None):
        
        self.name = name
        self.version = version
        self._phase = phase
        self._wave = wave
        self._parameters = np.array([1.])
        self._original_source = sncosmo.get_source( original_template_name, original_template_version)
        self._warp_spline = Spline2d(phase, wave, flux, kx=time_spline_degree,
                                    ky=3)
        self._zero_before = True # Assume this should always be the case here
        # What to interpolate to what? The warped spline should be a subset of the
        # full, using the same knots. 
        # So we should be able to interpolate the original source at the 
        # input phase, wave and then simple multiply?
        # Well, we have such course wavelength coverage in the warp template
        # that we more or less have to do it the other way around.
        # So we wish to find the original phase and wave which are within
        # limits

        # Create phases to use with the warped template
        self._phase = self._original_source._phase[( 
                ( self._original_source._phase >= phase.min() ) & 
                ( self._original_source._phase <= phase.max() )
                 )
        ]
        self._wave = self._original_source._wave[( 
                ( self._original_source._wave >= min(wave) ) & 
                ( self._original_source._wave <= max(wave) )
                 )
        ]
        # Template flux defined as original values divided by warp template
        flux = ( 
            self._original_source._flux( self._phase, self._wave ) *
            self._warp_spline( self._phase, self._wave )
        )
        print(flux)
        self._model_flux = Spline2d(self._phase, self._wave, flux, 
                                    kx=time_spline_degree, ky=3)

#        print(foo.shape)
#        print(foo2.shape)
#        myflux = foo / foo2
#        print(foo2)
#        for phase in self._original_source._phase:
#            print(phase)
#            newflux = self._original_source.flux( phase, self._original_source._wave ) * 
#                        self._warp_spline( ... )
#        self._model_flux = Spline2d(phase, wave, flux, kx=time_spline_degree,
#                                    ky=3)
#        self._zero_before = zero_before
    

In [ ]:
ws = WarpedTimeSeriesSource(
    phase=mdict['corrmodel']['phase'], 
    wave=mdict['corrmodel']['wave'], 
    flux=mdict['corrmodel']['flux'], original_template_name='snana-sdss004012')

In [ ]:
# Plot some fluxes
phases = [-8,-4,0,4,8,12,16,20,24,28,32,36,40,50,60,70]
bands = ['ztfg','ztfi','bessellv']
ws.bandflux(bands[2], 5)
for band in bands:
    plt.plot( phases, ws.bandflux(band, phases), label=band)
plt.legend()

Two ways of doing this:
- Do not interpolate. Load the base template info directly (circumvent source loader), use the new interpolator to modify the raw datapoints.
- Read the original source as interpolator. Extract fluxes at positions of the warp template, divide and create new interpolator.
Since we have already concluded we use the knots of the original template, the interpolation should be ok? So nothing to loose by using second method?

In [ ]:
dust = sncosmo.CCM89Dust()
wm = sncosmo.Model(source=ws,
                    effects=[dust],
                    effect_names=['host'],
                    effect_frames=['rest'])
wm.set(hostr_v = 3.1)
wm.set(z=z)
fitprop = ['t0', 'amplitude', 'hostebv']


In [ ]:
wresult, wfitted_model = sncosmo.fit_lc(
            tab, wm,
            fitprop,  # parameters of model to vary
        )


In [ ]:
fig = sncosmo.plot_lc(tab, model=wfitted_model, errors=wresult.errors)

In [ ]:
import extinction

In [ ]:
extinction.apply(extinction.ccm89(wave, ebv * r_v, r_v), flux)

In [ ]:
extinction.ccm89??

In [ ]:
from sncosmo import Source

class WarpSource( Source ):
    """
    Return a Source which match volume limited SNe from the 
    ZTF BTS program. This is based on an existing TimeSeriesSource
    template which is warped to match the ZTF g, R, (I) bands.    
    
    Parameters
    ----------
    source_class : str, optinal
        Base class of transients. One of: 
        'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T', 'SN Ib/c', 
        'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic','SLSN-II', 'SN IIn'
    ztf_name: str, optional
        Name of ZTF SN to source from.
        source_class OR ztf_name has to be given.
    name : str, optional
        Name of the model. Default is `None`.
    version : str, optional
        Version of the model. Default is `None`.

    """

    _param_names = ['amplitude']
    param_names_latex = ['A']


    
    def __init__(self, dirpath, randomness_parameter=None,
                 time_spline_degree=3, name=None, version=None):
        
        self.name = name
        self.version = version

        wave, phase, flux, peakmag = self.get_lightcurve( dirpath, randomness_parameter )
        
        self._phase = phase
        self._wave = wave

        self._model_flux = Spline2d(self._phase, self._wave, flux, 
                                    kx=time_spline_degree, ky=3)

        # Initial value for the amplitude parameter
        self._parameters = np.array([1.])
        # We then scale this to make sure the absolute magnitude is correct.
        self.set_peakmag(self, peakmag, 'bessellb', 'ab')

        
    def _flux(self, phase, wave):
        """
        This is the core method called by sncosmo
        """
        return self._parameters[0] * self._model_flux(phase, wave)


In [ ]:
m.source.get("_phase")